# CS781 Phase 3 - Rhetorical Role (RR): fetch, clean, audit, tokenise

**Before running**
1. Kaggle notebook settings: **Internet ON**. No GPU needed.
2. HF token in Kaggle Secrets (*Add-ons -> Secrets*), named `HF_TOKEN` or `HF` (both are tried). Your HF account must have accepted the conditions of `Exploration-Lab/CS-781-Capstone` (gated repo).
3. Add your shared **Phase-1 tokenizer dataset** as input.

**What it does** (only the Phase 3 RR files are downloaded)
- Finds the RR files in the repo by name, so the exact folder path does not matter.
- Cleans every sentence (odd spaces, zero-width and control characters, whitespace runs). **No sentence is ever dropped or merged**, so `len(predictions) == len(sentences)` always holds.
- Audits labels (null / wrong case / unknown labels), length mismatches, duplicate documents, train/dev/test leakage, repeated sentences with conflicting labels.
- Tokenises each sentence once with your tokenizer, then builds **window inputs** (sentence only, +-2 sentences) and **document chunks** (all sentences of a doc with a SEP after each).
- Verifies the saved files by reloading them.

**Output** (`/kaggle/working/rr_data_clean/`): cleaned `Train/` and `Test/` JSON, `label2id.json`, `class_weights.json`, `tokenized_rr.pkl`, `rr_utils.py`, `audit_report.json`, `meta.json`.

The dataset README says the files are *dynamic*, so a copy of the raw files you downloaded is also saved under `raw_snapshot/`.

## 1. Configuration

In [ ]:
import os
import re
import sys
import json
import glob
import time
import pickle
import shutil
import hashlib
import unicodedata
from collections import Counter

import numpy as np

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
DOMAINS = ["CL", "IT"]
SPLITS = ["train", "dev", "test"]

# Where things live. The env vars are only for local testing; on Kaggle leave them alone.
LOCAL_DATA_DIR = os.environ.get("RR_LOCAL_DATA_DIR", None)   # folder holding the 7 RR files (any depth)
TOKENIZER_DIR = os.environ.get("RR_TOKENIZER_DIR", None)     # None = search /kaggle/input
OUT_DIR = os.environ.get("RR_OUT_DIR", "/kaggle/working/rr_data_clean")

# Cleaning
NORMALISE_PUNCT = False     # smart quotes / dashes -> ascii. Off: unknown how Phase-1 text was normalised.
EMPTY_PLACEHOLDER = "."     # used if a sentence is empty after cleaning (never drop a sentence)

# Tokenisation
MAX_SENT_TOK = 160          # cap on tokens kept per sentence
WINDOWS_TO_BUILD = [0, 2]   # 0 = sentence only, 2 = +-2 sentences of context (other windows: rr_utils on the fly)
BUDGET_PREV = 192           # max tokens of previous-sentence context
BUDGET_NEXT = 96            # max tokens of next-sentence context
DOC_CHUNK_LEN = 1024        # max tokens per document chunk (<= smallest Phase-1 context length)
DOC_CHUNK_CTX = 2           # sentences repeated from the previous chunk as context

DOMAIN_TEXT = {"CL": "Corporate Law.", "IT": "Information Technology."}
DOMAIN_ID = {"CL": 0, "IT": 1}

EXPECTED_LABELS = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent",
    "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled",
    "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None",
]

os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

## 2. Hugging Face login

In [ ]:
HF_TOKEN = None

if LOCAL_DATA_DIR is None:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in ["HF_TOKEN", "HF"]:
            if HF_TOKEN is None:
                try:
                    HF_TOKEN = secrets.get_secret(secret_name)
                    print("Using Kaggle secret:", secret_name)
                except Exception:
                    pass
    except ImportError:
        print("kaggle_secrets not available (not on Kaggle).")

    if HF_TOKEN is None:
        HF_TOKEN = os.environ.get("HF_TOKEN", None)

    if HF_TOKEN is None:
        raise RuntimeError("No HF token found. Add a Kaggle secret named HF_TOKEN (or HF).")

    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("Local data mode, skipping Hugging Face login.")

## 3. Find and download the RR files

In [ ]:
WANTED = [
    "CL_train.json", "CL_dev.json", "IT_train.json", "IT_dev.json",
    "label_vocab.json", "CL_test.json", "IT_test.json",
]

def basename_of(path):
    return os.path.basename(path)

found = {}

if LOCAL_DATA_DIR is not None:
    all_paths = glob.glob(os.path.join(LOCAL_DATA_DIR, "**", "*.json"), recursive=True)
    for path in all_paths:
        name = basename_of(path)
        if name in WANTED:
            found[name] = path
else:
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=HF_TOKEN)
    repo_files = api.list_repo_files(HF_DATASET_REPO, repo_type="dataset")
    print("Files in repo:", len(repo_files))

    # keep only files that look like Phase 3 / RR
    candidates = []
    for path in repo_files:
        low = path.lower()
        if "phase_3" in low or "phase3" in low or "/rr/" in low or low.startswith("rr/"):
            candidates.append(path)

    for name in WANTED:
        matches = []
        for path in candidates:
            if basename_of(path) == name:
                matches.append(path)
        if len(matches) == 1:
            found[name] = matches[0]
        elif len(matches) > 1:
            print("More than one match for", name, "->", matches, "(using the first)")
            found[name] = matches[0]

    missing = []
    for name in WANTED:
        if name not in found:
            missing.append(name)
    if len(missing) > 0:
        print("MISSING in repo:", missing)
        print("Phase 3 / RR looking paths I did see:")
        for path in candidates:
            print("  ", path)
        raise FileNotFoundError("Could not find: " + ", ".join(missing))

    for name in WANTED:
        found[name] = hf_hub_download(
            repo_id=HF_DATASET_REPO, filename=found[name], repo_type="dataset", token=HF_TOKEN
        )

for name in WANTED:
    if name not in found:
        raise FileNotFoundError(name + " not found under " + str(LOCAL_DATA_DIR))
    print(name, "->", found[name])

# keep a snapshot of the raw files (dataset is said to be dynamic)
snap_dir = os.path.join(OUT_DIR, "raw_snapshot")
os.makedirs(snap_dir, exist_ok=True)
for name in WANTED:
    shutil.copy(found[name], os.path.join(snap_dir, name))
print("Raw snapshot saved to", snap_dir)

## 4. Load labels and documents

In [ ]:
def read_json(path):
    f = open(path, "r", encoding="utf-8")
    data = json.load(f)
    f.close()
    return data


def load_label_vocab(path):
    raw = read_json(path)
    label2id = {}

    if isinstance(raw, list):
        index = 0
        for label in raw:
            label2id[label] = index
            index = index + 1
    elif isinstance(raw, dict):
        keys = list(raw.keys())
        first_value = raw[keys[0]]
        if isinstance(first_value, int):
            for label in raw:
                label2id[label] = int(raw[label])
        else:
            for key in raw:
                label2id[raw[key]] = int(key)
    return label2id


TEXT_KEYS = ["text", "sentences", "sentence", "sents", "content"]
ID_KEYS = ["id", "doc_id", "document_id", "docid", "name", "file", "filename"]
WRAPPER_KEYS = ["data", "documents", "docs", "test", "train", "dev", "items", "examples", "instances", "judgments", "cases"]


def as_sentence_list(value):
    # a list of strings, or a list of {"text"/"sentence": str} dicts
    if not isinstance(value, list) or len(value) == 0:
        return None
    sentences = []
    for item in value:
        if isinstance(item, str):
            sentences.append(item)
        elif isinstance(item, dict):
            got = None
            for key in ["text", "sentence", "sent"]:
                if key in item and isinstance(item[key], str):
                    got = item[key]
                    break
            if got is None:
                return None
            sentences.append(got)
        else:
            return None
    return sentences


def doc_from_item(item, fallback_id):
    if isinstance(item, list):
        sentences = as_sentence_list(item)
        if sentences is None:
            return None
        return {"id": fallback_id, "text": sentences}
    if isinstance(item, dict):
        doc = dict(item)
        sentences = None
        for key in TEXT_KEYS:
            if key in doc:
                sentences = as_sentence_list(doc[key])
                if sentences is not None:
                    break
        if sentences is None:
            if "text" in doc and isinstance(doc["text"], list) and len(doc["text"]) == 0:
                sentences = []
            else:
                return None
        doc["text"] = sentences
        doc_id = None
        for key in ID_KEYS:
            if key in doc and doc[key] is not None:
                doc_id = doc[key]
                break
        doc["id"] = str(doc_id) if doc_id is not None else fallback_id
        return doc
    return None


def describe_raw(raw, n_chars=300):
    if isinstance(raw, dict):
        keys = list(raw.keys())
        first = raw[keys[0]] if len(keys) > 0 else None
        return "dict with %d keys, first keys %s, first value type %s: %s" % (
            len(keys), keys[:5], type(first).__name__, repr(first)[:n_chars])
    if isinstance(raw, list):
        first = raw[0] if len(raw) > 0 else None
        return "list of %d items, first item type %s: %s" % (len(raw), type(first).__name__, repr(first)[:n_chars])
    return type(raw).__name__ + ": " + repr(raw)[:n_chars]


def to_doc_list(raw, tag):
    # accepted layouts: list of docs | one doc | {doc_id: doc} | {doc_id: [sentences]} | {"data": ...} wrapper
    for _ in range(3):
        if isinstance(raw, dict) and len(raw) == 1:
            only_key = list(raw.keys())[0]
            if only_key in WRAPPER_KEYS and isinstance(raw[only_key], (list, dict)):
                raw = raw[only_key]
                continue
        break

    pairs = []
    if isinstance(raw, dict):
        if any(k in raw for k in TEXT_KEYS) and as_sentence_list(raw.get("text", raw.get("sentences"))) is not None:
            pairs.append((tag + "_0", raw))
        else:
            for key in raw:
                pairs.append((str(key), raw[key]))
    elif isinstance(raw, list):
        for index in range(len(raw)):
            pairs.append((tag + "_" + str(index), raw[index]))

    docs = []
    skipped = 0
    for fallback_id, item in pairs:
        doc = doc_from_item(item, fallback_id)
        if doc is None:
            skipped += 1
            continue
        docs.append(doc)
    if skipped > 0:
        print("WARNING:", tag, ": could not read", skipped, "of", len(pairs), "entries as documents")
    if skipped > 0 and "test" in tag:
        raise ValueError(tag + ": " + str(skipped) + " test entries could not be read. Structure: " + describe_raw(raw))
    return docs


label2id = load_label_vocab(found["label_vocab.json"])
id2label = {}
for label in label2id:
    id2label[label2id[label]] = label

print("labels in vocab:", len(label2id))
print(sorted(label2id.items(), key=lambda kv: kv[1]))

vocab_set = set(label2id.keys())
expected_set = set(EXPECTED_LABELS)
if vocab_set != expected_set:
    print("WARNING: label_vocab differs from the PDF list.")
    print("  only in vocab:", sorted(vocab_set - expected_set))
    print("  only in PDF  :", sorted(expected_set - vocab_set))

raw_docs = {}
for domain in DOMAINS:
    raw_docs[domain] = {}
    for split in SPLITS:
        name = domain + "_" + split + ".json"
        raw = read_json(found[name])
        raw_docs[domain][split] = to_doc_list(raw, name)
        n_bytes = os.path.getsize(found[name])
        print(name, ":", n_bytes, "bytes | docs =", len(raw_docs[domain][split]))
        if len(raw_docs[domain][split]) == 0:
            print("   structure:", describe_raw(raw))
            if n_bytes > 1000:
                raise RuntimeError(name + " is " + str(n_bytes) + " bytes but no document could be read from it. "
                                   "Send the 'structure' line above so the reader can be fixed.")
            if split == "test":
                print("   >>> " + name + " is EMPTY in the repo (" + str(n_bytes) + " bytes): the test set is not released in this file.")

## 5. Clean sentences

Character-level fixes only. Sentences are never dropped, merged or reordered.

In [ ]:
ZERO_WIDTH = ["​", "‌", "‍", "⁠", "﻿", "­"]

PUNCT_MAP = {
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "−": "-", "…": "...",
}

clean_stats = Counter()


def clean_sentence(s):
    if s is None:
        clean_stats["none_sentence"] += 1
        s = ""
    if not isinstance(s, str):
        clean_stats["non_string_sentence"] += 1
        s = str(s)

    original = s
    s = unicodedata.normalize("NFC", s)

    # remove zero-width / soft hyphen
    for zw in ZERO_WIDTH:
        if zw in s:
            clean_stats["zero_width_removed"] += 1
            s = s.replace(zw, "")

    # turn odd spaces and control characters into plain spaces, drop other format chars
    out = []
    for ch in s:
        cat = unicodedata.category(ch)
        if ch == " ":
            out.append(ch)
        elif cat == "Zs":
            clean_stats["odd_space_replaced"] += 1
            out.append(" ")
        elif cat in ("Cc", "Zl", "Zp"):
            clean_stats["control_or_newline_replaced"] += 1
            out.append(" ")
        elif cat == "Cf":
            clean_stats["format_char_removed"] += 1
        else:
            out.append(ch)
    s = "".join(out)

    if "�" in s:
        clean_stats["replacement_char_present"] += 1

    if NORMALISE_PUNCT:
        for bad in PUNCT_MAP:
            s = s.replace(bad, PUNCT_MAP[bad])

    s = " ".join(s.split())

    if s == "":
        clean_stats["empty_after_clean"] += 1
        s = EMPTY_PLACEHOLDER

    if s != original:
        clean_stats["sentences_changed"] += 1
    clean_stats["sentences_total"] += 1
    return s


# quick self-test
test_in = "A B​C\t D\n\n E"
test_out = clean_sentence(test_in)
print(repr(test_in), "->", repr(test_out))
assert test_out == "A BC D E"
clean_stats.clear()

## 6. Audit and repair

- Labels are repaired where safe: spaces, wrong case, and JSON `null` become the string label `None` (a real label here).
- A train/dev document with a length mismatch or an unknown label is dropped (alignment cannot be trusted). **Test documents are never dropped.**
- Duplicate documents are removed from train, and from dev when they also appear in train. Test overlap is only reported.

In [ ]:
lower_to_label = {}
for label in label2id:
    lower_to_label[label.lower()] = label

audit = {"dropped_docs": [], "label_repairs": Counter(), "notes": []}


def repair_label(raw_label):
    if raw_label is None:
        if "None" in label2id:
            audit["label_repairs"]["null_to_None_label"] += 1
            return "None"
        return None
    text = str(raw_label).strip()
    if text in label2id:
        if text != raw_label:
            audit["label_repairs"]["stripped_spaces"] += 1
        return text
    low = text.lower()
    if low in lower_to_label:
        audit["label_repairs"]["case_fixed"] += 1
        return lower_to_label[low]
    return None


def doc_hash(sentences):
    h = hashlib.md5()
    for s in sentences:
        h.update(s.encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()


clean_docs = {}

for domain in DOMAINS:
    clean_docs[domain] = {}
    seen_train_hash = {}

    for split in SPLITS:
        out_docs = []
        used_ids = Counter()
        seen_hash_in_split = {}
        n_dup_within = 0
        n_dup_vs_train = 0
        tag = domain + "_" + split

        for doc in raw_docs[domain][split]:
            doc_id = doc["id"]
            raw_text = doc.get("text", None)

            # ---- structure checks ----
            if not isinstance(raw_text, list):
                if split == "test":
                    raise ValueError("Test doc " + doc_id + " has no sentence list.")
                audit["dropped_docs"].append([tag, doc_id, "text_not_a_list"])
                continue

            if len(raw_text) == 0 and split != "test":
                audit["dropped_docs"].append([tag, doc_id, "zero_sentences"])
                continue

            sentences = []
            for s in raw_text:
                sentences.append(clean_sentence(s))

            label_ids = None
            if split != "test":
                raw_labels = doc.get("labels", None)
                if not isinstance(raw_labels, list):
                    audit["dropped_docs"].append([tag, doc_id, "labels_missing"])
                    continue
                if len(raw_labels) != len(sentences):
                    audit["dropped_docs"].append(
                        [tag, doc_id, "length_mismatch " + str(len(sentences)) + " vs " + str(len(raw_labels))]
                    )
                    continue
                label_ids = []
                bad_label = None
                for raw_label in raw_labels:
                    fixed = repair_label(raw_label)
                    if fixed is None:
                        bad_label = raw_label
                        break
                    label_ids.append(label2id[fixed])
                if bad_label is not None:
                    audit["dropped_docs"].append([tag, doc_id, "unknown_label " + repr(bad_label)])
                    continue

            # ---- duplicates ----
            h = doc_hash(sentences)
            if split == "train":
                if h in seen_hash_in_split:
                    n_dup_within += 1
                    audit["dropped_docs"].append([tag, doc_id, "duplicate_of_" + seen_hash_in_split[h]])
                    continue
                seen_hash_in_split[h] = doc_id
                seen_train_hash[h] = doc_id
            elif split == "dev":
                if h in seen_train_hash:
                    n_dup_vs_train += 1
                    audit["dropped_docs"].append([tag, doc_id, "dev_doc_also_in_train_" + seen_train_hash[h]])
                    continue
                if h in seen_hash_in_split:
                    n_dup_within += 1
                    audit["dropped_docs"].append([tag, doc_id, "duplicate_of_" + seen_hash_in_split[h]])
                    continue
                seen_hash_in_split[h] = doc_id
            else:
                if h in seen_train_hash:
                    n_dup_vs_train += 1   # report only

            # ---- unique id ----
            used_ids[doc_id] += 1
            final_id = doc_id
            if used_ids[doc_id] > 1:
                if split == "test":
                    raise ValueError("Duplicate test doc id: " + doc_id)
                final_id = doc_id + "__dup" + str(used_ids[doc_id])
                audit["notes"].append(tag + ": duplicate id " + doc_id + " renamed " + final_id)

            out_docs.append({
                "id": final_id,
                "sentences": sentences,
                "label_ids": label_ids,
                "hash": h,
            })

        clean_docs[domain][split] = out_docs
        print(tag, ": kept", len(out_docs), "of", len(raw_docs[domain][split]),
              "| dup within:", n_dup_within, "| vs train:", n_dup_vs_train)

print()
print("Cleaning stats:", dict(clean_stats))
print("Label repairs :", dict(audit["label_repairs"]))
print("Dropped docs  :", len(audit["dropped_docs"]))
for row in audit["dropped_docs"][:20]:
    print("  ", row)

# test docs must all survive with the same sentence counts
for domain in DOMAINS:
    n_raw = len(raw_docs[domain]["test"])
    n_clean = len(clean_docs[domain]["test"])
    assert n_raw == n_clean, "test documents were lost in " + domain
    for k in range(n_raw):
        assert len(raw_docs[domain]["test"][k]["text"]) == len(clean_docs[domain]["test"][k]["sentences"])
print("Test documents and sentence counts preserved.")

### Label statistics, leakage and noise checks

In [ ]:
num_labels = len(label2id)
report = {"domains": {}}
transition_counts = {}


def sentence_key(s):
    return " ".join(s.lower().split())


for domain in DOMAINS:
    dom_report = {}
    train_docs = clean_docs[domain]["train"]
    dev_docs = clean_docs[domain]["dev"]
    test_docs = clean_docs[domain]["test"]

    # ---- label counts + mean relative position ----
    for split in ["train", "dev"]:
        counts = np.zeros(num_labels, dtype=np.int64)
        pos_sum = np.zeros(num_labels, dtype=np.float64)
        total = 0
        for doc in clean_docs[domain][split]:
            n = len(doc["sentences"])
            for i in range(n):
                lab = doc["label_ids"][i]
                counts[lab] += 1
                pos_sum[lab] += i / max(n - 1, 1)
                total += 1
        print("\n==", domain, split, "| docs:", len(clean_docs[domain][split]), "| sentences:", total)
        rows = []
        for lab in range(num_labels):
            if counts[lab] > 0:
                mean_pos = pos_sum[lab] / counts[lab]
            else:
                mean_pos = float("nan")
            rows.append((id2label[lab], int(counts[lab]), 100.0 * counts[lab] / max(total, 1), mean_pos))
        rows.sort(key=lambda r: -r[1])
        for r in rows:
            print("   %-24s %7d  %6.2f%%   mean rel.pos %.2f" % (r[0], r[1], r[2], r[3]))
        dom_report[split + "_label_counts"] = {}
        for r in rows:
            dom_report[split + "_label_counts"][r[0]] = r[1]
        missing_labels = []
        for r in rows:
            if r[1] == 0:
                missing_labels.append(r[0])
        if len(missing_labels) > 0:
            print("   labels with ZERO examples:", missing_labels)
        dom_report[split + "_missing_labels"] = missing_labels

    # ---- label transitions (train) ----
    trans = np.zeros((num_labels, num_labels), dtype=np.int64)
    for doc in train_docs:
        for i in range(len(doc["label_ids"]) - 1):
            trans[doc["label_ids"][i], doc["label_ids"][i + 1]] += 1
    transition_counts[domain] = trans.tolist()
    same = 0
    for lab in range(num_labels):
        same += trans[lab, lab]
    print("   P(next label == current label) in train: %.3f" % (same / max(trans.sum(), 1)))

    # ---- repeated sentences / conflicting labels (train) ----
    sent_to_labels = {}
    for doc in train_docs:
        for i in range(len(doc["sentences"])):
            key = sentence_key(doc["sentences"][i])
            if key not in sent_to_labels:
                sent_to_labels[key] = Counter()
            sent_to_labels[key][doc["label_ids"][i]] += 1
    n_repeated = 0
    n_conflict = 0
    for key in sent_to_labels:
        total_count = sum(sent_to_labels[key].values())
        if total_count > 1:
            n_repeated += 1
            if len(sent_to_labels[key]) > 1:
                n_conflict += 1
    print("   train: distinct sentences repeated:", n_repeated,
          "| of those with conflicting labels:", n_conflict)
    dom_report["train_repeated_sentences"] = n_repeated
    dom_report["train_repeated_with_conflict"] = n_conflict

    # ---- sentence-level overlap dev/test vs train ----
    for split in ["dev", "test"]:
        n_total = 0
        n_in_train = 0
        n_agree = 0
        for doc in clean_docs[domain][split]:
            for i in range(len(doc["sentences"])):
                n_total += 1
                key = sentence_key(doc["sentences"][i])
                if key in sent_to_labels:
                    n_in_train += 1
                    if split == "dev":
                        majority = sent_to_labels[key].most_common(1)[0][0]
                        if majority == doc["label_ids"][i]:
                            n_agree += 1
        msg = "   " + split + ": sentences also seen verbatim in train: " + str(n_in_train) + "/" + str(n_total)
        if split == "dev" and n_in_train > 0:
            msg = msg + " | train-majority label correct for " + str(n_agree)
        print(msg)
        dom_report[split + "_sentences_seen_in_train"] = n_in_train
        dom_report[split + "_sentences_total"] = n_total

    # ---- document length ----
    lens = []
    for doc in train_docs:
        lens.append(len(doc["sentences"]))
    if len(lens) > 0:
        lens.sort()
        print("   sentences/doc (train): median", lens[len(lens) // 2], "| max", lens[-1])

    report["domains"][domain] = dom_report

audit_report = {
    "cleaning": dict(clean_stats),
    "label_repairs": dict(audit["label_repairs"]),
    "dropped_docs": audit["dropped_docs"],
    "notes": audit["notes"],
    "domains": report["domains"],
    "label_transitions_train": transition_counts,
}

## 7. Class weights and cleaned files

In [ ]:
def make_class_weights(domain_list):
    counts = np.zeros(num_labels, dtype=np.float64)
    for domain in domain_list:
        for doc in clean_docs[domain]["train"]:
            for lab in doc["label_ids"]:
                counts[lab] += 1
    total = counts.sum()

    inverse = np.zeros(num_labels)
    inverse_sqrt = np.zeros(num_labels)
    effective = np.zeros(num_labels)
    beta = 0.999
    for lab in range(num_labels):
        if counts[lab] > 0:
            inverse[lab] = total / (num_labels * counts[lab])
            inverse_sqrt[lab] = np.sqrt(inverse[lab])
            effective[lab] = (1.0 - beta) / (1.0 - beta ** counts[lab])
    # rescale effective weights so the mean over seen classes is 1
    seen = counts > 0
    effective = effective * (seen.sum() / effective.sum())
    inverse_sqrt = inverse_sqrt * (seen.sum() / inverse_sqrt.sum())

    return {
        "counts": counts.tolist(),
        "inverse": inverse.tolist(),
        "inverse_sqrt": inverse_sqrt.tolist(),
        "effective_number": effective.tolist(),
    }


class_weights = {}
for domain in DOMAINS:
    class_weights[domain] = make_class_weights([domain])
class_weights["ALL"] = make_class_weights(DOMAINS)

with open(os.path.join(OUT_DIR, "class_weights.json"), "w") as f:
    json.dump(class_weights, f, indent=2)
with open(os.path.join(OUT_DIR, "label2id.json"), "w") as f:
    json.dump(label2id, f, indent=2)

# cleaned docs in the ORIGINAL format
for domain in DOMAINS:
    for split in SPLITS:
        if split == "test":
            sub = "Test"
        else:
            sub = "Train"
        os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
        out_list = []
        for doc in clean_docs[domain][split]:
            item = {"id": doc["id"], "text": doc["sentences"]}
            if doc["label_ids"] is not None:
                names = []
                for lab in doc["label_ids"]:
                    names.append(id2label[lab])
                item["labels"] = names
            out_list.append(item)
        path = os.path.join(OUT_DIR, sub, domain + "_" + split + ".json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump(out_list, f, ensure_ascii=False)
shutil.copy(os.path.join(OUT_DIR, "label2id.json"), os.path.join(OUT_DIR, "Train", "label_vocab.json"))
print("Cleaned files written to", OUT_DIR)

## 8. Load the Phase-1 tokenizer

In [ ]:
from transformers import AutoTokenizer


def find_tokenizer_dir():
    if TOKENIZER_DIR is not None:
        return TOKENIZER_DIR
    paths = glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
    folders = []
    for p in paths:
        folder = os.path.dirname(p)
        if "checkpoint-" not in folder:
            folders.append(folder)
    folders = sorted(set(folders))
    if len(folders) == 0:
        raise FileNotFoundError("No tokenizer.json under /kaggle/input. Attach your tokenizer dataset (Add Input).")
    for folder in folders:
        if os.path.exists(os.path.join(folder, "trained_on.json")):
            return folder
    return folders[0]


tok_dir = find_tokenizer_dir()
tokenizer = AutoTokenizer.from_pretrained(tok_dir)
tokenizer.model_max_length = 10 ** 9
print("Tokenizer:", tok_dir, "| vocab size:", len(tokenizer))


def find_sep_id(tok):
    if tok.eos_token_id is not None:
        return int(tok.eos_token_id)
    for name in ["<eos>", "</s>", "<|endoftext|>", "<EOS>"]:
        tid = tok.convert_tokens_to_ids(name)
        if tid is not None and tid != tok.unk_token_id:
            return int(tid)
    raise ValueError("Could not find an EOS-like token to use as separator.")


SEP_ID = find_sep_id(tokenizer)
if tokenizer.pad_token_id is not None:
    PAD_ID = int(tokenizer.pad_token_id)
else:
    PAD_ID = SEP_ID
print("SEP id (EOS, same separator Phase-1 used between documents):", SEP_ID, "| PAD id:", PAD_ID)


def encode_text(text):
    out = tokenizer(text, add_special_tokens=False)
    return list(out["input_ids"])


HEADER_IDS = {}
for domain in DOMAINS:
    HEADER_IDS[domain] = encode_text(DOMAIN_TEXT[domain])
BEFORE_IDS = encode_text("Preceding text:")
AFTER_IDS = encode_text("Following text:")
SENT_MARKER_IDS = encode_text("Target sentence:")
print("header ids:", HEADER_IDS)

## 9. Input-building helpers (saved so later notebooks reuse them)

In [ ]:
%%writefile rr_utils.py
"""Helpers for CS781 Phase 3 (Rhetorical Role) input building.

Written by the data-prep notebook so every later notebook builds inputs the
same way.  Only needs numpy.

Two input formats:

1) Window format (one example per sentence)
   [SEP] domain [SEP] BEFORE prev sentences [SEP] AFTER next sentences [SEP] SENT target
   The target sentence is LAST, so a decoder (causal) model can pool the last
   token or mean-pool ids[target_start:] and still see all the context.

2) Chunk format (one example per piece of a document)
   [SEP] domain  s1 [SEP]  s2 [SEP]  s3 [SEP] ...
   Every sentence ends with a SEP token; read the hidden state at that SEP
   position to classify the sentence.  Context sentences copied from the
   previous chunk get label -100.
"""

import numpy as np

IGNORE_LABEL = -100


def cap_ids(ids, max_tok):
    ids = list(ids)
    if len(ids) > max_tok:
        ids = ids[:max_tok]
    return ids


def build_window_example(sent_ids, i, window, header_ids, before_ids, after_ids,
                         sent_marker_ids, sep_id, max_sent_tok,
                         budget_prev, budget_next):
    """Returns (input_ids as list, target_start)."""
    n = len(sent_ids)
    target = cap_ids(sent_ids[i], max_sent_tok)

    # ---- previous sentences (nearest first, then reversed) ----
    prev_pieces = []
    prev_left = budget_prev
    j = i - 1
    stop = i - window
    if stop < 0:
        stop = 0
    while j >= stop:
        piece = cap_ids(sent_ids[j], max_sent_tok)
        if len(piece) <= prev_left:
            prev_pieces.append(piece)
            prev_left = prev_left - len(piece) - 1
        else:
            if prev_left > 16:
                # keep the END of the sentence (closest to the target)
                piece = piece[len(piece) - prev_left:]
                prev_pieces.append(piece)
            break
        j = j - 1
    prev_pieces.reverse()

    # ---- next sentences (nearest first) ----
    next_pieces = []
    next_left = budget_next
    j = i + 1
    stop = i + window + 1
    if stop > n:
        stop = n
    while j < stop:
        piece = cap_ids(sent_ids[j], max_sent_tok)
        if len(piece) <= next_left:
            next_pieces.append(piece)
            next_left = next_left - len(piece) - 1
        else:
            if next_left > 16:
                # keep the START of the sentence (closest to the target)
                piece = piece[:next_left]
                next_pieces.append(piece)
            break
        j = j + 1

    # ---- assemble ----
    ids = []
    ids.append(sep_id)
    for t in header_ids:
        ids.append(t)

    if len(prev_pieces) > 0:
        ids.append(sep_id)
        for t in before_ids:
            ids.append(t)
        first = True
        for piece in prev_pieces:
            if not first:
                ids.append(sep_id)
            for t in piece:
                ids.append(t)
            first = False

    if len(next_pieces) > 0:
        ids.append(sep_id)
        for t in after_ids:
            ids.append(t)
        first = True
        for piece in next_pieces:
            if not first:
                ids.append(sep_id)
            for t in piece:
                ids.append(t)
            first = False

    ids.append(sep_id)
    for t in sent_marker_ids:
        ids.append(t)

    target_start = len(ids)
    for t in target:
        ids.append(t)

    return ids, target_start


def build_doc_chunks(sent_ids, labels, header_ids, sep_id, max_len,
                     ctx_sents, max_sent_tok):
    """Split one document into chunks of at most max_len tokens.

    Returns a list of dicts with:
      input_ids : list of token ids
      end_pos   : position of the SEP that ends each sentence in the chunk
      sent_idx  : index of that sentence inside the document
      is_target : True if the sentence belongs to this chunk (False = context only)
      labels    : label id for target sentences, -100 for context / unlabeled
    labels may be None (test data).
    """
    n = len(sent_ids)
    header = [sep_id]
    for t in header_ids:
        header.append(t)

    if len(header) + max_sent_tok + 1 > max_len:
        raise ValueError("max_len too small for max_sent_tok")

    chunks = []
    start = 0
    while start < n:
        # context sentences taken from before `start`
        ctx_start = start - ctx_sents
        if ctx_start < 0:
            ctx_start = 0
        ctx_list = []
        for j in range(ctx_start, start):
            ctx_list.append(j)

        first_len = len(cap_ids(sent_ids[start], max_sent_tok)) + 1
        used = len(header) + first_len
        for j in ctx_list:
            used = used + len(cap_ids(sent_ids[j], max_sent_tok)) + 1
        while used > max_len and len(ctx_list) > 0:
            dropped = ctx_list.pop(0)
            used = used - len(cap_ids(sent_ids[dropped], max_sent_tok)) - 1

        ids = list(header)
        end_pos = []
        sent_idx = []
        is_target = []
        chunk_labels = []

        for j in ctx_list:
            piece = cap_ids(sent_ids[j], max_sent_tok)
            for t in piece:
                ids.append(t)
            ids.append(sep_id)
            end_pos.append(len(ids) - 1)
            sent_idx.append(j)
            is_target.append(False)
            chunk_labels.append(IGNORE_LABEL)

        j = start
        while j < n:
            piece = cap_ids(sent_ids[j], max_sent_tok)
            if len(ids) + len(piece) + 1 > max_len:
                break
            for t in piece:
                ids.append(t)
            ids.append(sep_id)
            end_pos.append(len(ids) - 1)
            sent_idx.append(j)
            is_target.append(True)
            if labels is None:
                chunk_labels.append(IGNORE_LABEL)
            else:
                chunk_labels.append(int(labels[j]))
            j = j + 1

        chunk = {}
        chunk["input_ids"] = np.array(ids, dtype=np.int32)
        chunk["end_pos"] = np.array(end_pos, dtype=np.int32)
        chunk["sent_idx"] = np.array(sent_idx, dtype=np.int32)
        chunk["is_target"] = np.array(is_target, dtype=bool)
        chunk["labels"] = np.array(chunk_labels, dtype=np.int64)
        chunks.append(chunk)

        start = j

    return chunks


def pad_sequences(list_of_ids, pad_id, left_pad=False):
    """Pad a list of id lists. Returns (ids, attention_mask) as numpy int64."""
    longest = 0
    for ids in list_of_ids:
        if len(ids) > longest:
            longest = len(ids)

    batch = len(list_of_ids)
    out_ids = np.full((batch, longest), pad_id, dtype=np.int64)
    out_mask = np.zeros((batch, longest), dtype=np.int64)

    for row in range(batch):
        ids = list_of_ids[row]
        length = len(ids)
        if left_pad:
            out_ids[row, longest - length:] = ids
            out_mask[row, longest - length:] = 1
        else:
            out_ids[row, :length] = ids
            out_mask[row, :length] = 1

    return out_ids, out_mask

In [ ]:
sys.path.insert(0, os.getcwd())
import importlib
import rr_utils
importlib.reload(rr_utils)
shutil.copy("rr_utils.py", os.path.join(OUT_DIR, "rr_utils.py"))
print("rr_utils imported and copied to", OUT_DIR)

## 10. Tokenise sentences, build window inputs and document chunks

In [ ]:
def encode_many(texts, batch_size=2000):
    all_ids = []
    start = 0
    while start < len(texts):
        batch = texts[start:start + batch_size]
        out = tokenizer(batch, add_special_tokens=False,
                        return_attention_mask=False, return_token_type_ids=False)
        for ids in out["input_ids"]:
            all_ids.append(ids)
        start = start + batch_size
    return all_ids


tok_stats = {}
roundtrip_bad = 0
roundtrip_checked = 0
t0 = time.time()

for domain in DOMAINS:
    for split in SPLITS:
        docs = clean_docs[domain][split]

        flat_texts = []
        for doc in docs:
            for s in doc["sentences"]:
                flat_texts.append(s)

        flat_ids = encode_many(flat_texts)

        # round trip check on (up to) the first 5000 sentences of each file
        limit = min(5000, len(flat_texts))
        for k in range(limit):
            back = tokenizer.decode(flat_ids[k], skip_special_tokens=False, clean_up_tokenization_spaces=False)
            roundtrip_checked += 1
            if back != flat_texts[k]:
                roundtrip_bad += 1

        n_trunc = 0
        lengths = []
        k = 0
        for doc in docs:
            doc_ids = []
            for s in doc["sentences"]:
                ids = flat_ids[k]
                lengths.append(len(ids))
                if len(ids) > MAX_SENT_TOK:
                    n_trunc += 1
                    ids = ids[:MAX_SENT_TOK]
                doc_ids.append(np.array(ids, dtype=np.int32))
                k += 1
            doc["sent_ids"] = doc_ids

        lengths_sorted = sorted(lengths)
        tok_stats[domain + "_" + split] = {
            "sentences": len(lengths),
            "median_tokens": lengths_sorted[len(lengths_sorted) // 2] if len(lengths) > 0 else 0,
            "p95_tokens": lengths_sorted[int(len(lengths_sorted) * 0.95)] if len(lengths) > 0 else 0,
            "max_tokens": lengths_sorted[-1] if len(lengths) > 0 else 0,
            "truncated_sentences": n_trunc,
        }
        print(domain, split, tok_stats[domain + "_" + split])

print("Round trip decode(encode(s)) == s : bad", roundtrip_bad, "of", roundtrip_checked)
print("Sentence tokenisation took %.1fs" % (time.time() - t0))

In [ ]:
tokenized = {}

for domain in DOMAINS:
    tokenized[domain] = {}
    for split in SPLITS:
        docs = clean_docs[domain][split]
        entry = {}

        # ---------- documents (compact: per-sentence token ids) ----------
        doc_list = []
        for doc in docs:
            item = {}
            item["id"] = doc["id"]
            item["sentences"] = doc["sentences"]
            item["sent_ids"] = doc["sent_ids"]
            if doc["label_ids"] is None:
                item["labels"] = None
            else:
                item["labels"] = np.array(doc["label_ids"], dtype=np.int64)
            doc_list.append(item)
        entry["docs"] = doc_list

        # ---------- window inputs (one row per sentence) ----------
        entry["windows"] = {}
        for window in WINDOWS_TO_BUILD:
            rows_ids = []
            rows_target_start = []
            rows_labels = []
            rows_doc = []
            rows_sent = []
            rows_relpos = []
            for d in range(len(docs)):
                doc = docs[d]
                n = len(doc["sent_ids"])
                for i in range(n):
                    ids, target_start = rr_utils.build_window_example(
                        doc["sent_ids"], i, window,
                        HEADER_IDS[domain], BEFORE_IDS, AFTER_IDS, SENT_MARKER_IDS,
                        SEP_ID, MAX_SENT_TOK, BUDGET_PREV, BUDGET_NEXT,
                    )
                    rows_ids.append(np.array(ids, dtype=np.int32))
                    rows_target_start.append(target_start)
                    if doc["label_ids"] is None:
                        rows_labels.append(-1)
                    else:
                        rows_labels.append(doc["label_ids"][i])
                    rows_doc.append(d)
                    rows_sent.append(i)
                    rows_relpos.append(i / max(n - 1, 1))
            entry["windows"][window] = {
                "input_ids": rows_ids,
                "target_start": np.array(rows_target_start, dtype=np.int32),
                "labels": np.array(rows_labels, dtype=np.int64),
                "doc_idx": np.array(rows_doc, dtype=np.int32),
                "sent_idx": np.array(rows_sent, dtype=np.int32),
                "rel_pos": np.array(rows_relpos, dtype=np.float32),
            }

        # ---------- document chunks ----------
        chunk_list = []
        for d in range(len(docs)):
            doc = docs[d]
            chunks = rr_utils.build_doc_chunks(
                doc["sent_ids"], doc["label_ids"], HEADER_IDS[domain],
                SEP_ID, DOC_CHUNK_LEN, DOC_CHUNK_CTX, MAX_SENT_TOK,
            )
            for chunk in chunks:
                chunk["doc_idx"] = d
                chunk_list.append(chunk)
        entry["chunks"] = chunk_list

        tokenized[domain][split] = entry

        n_sent = 0
        for doc in docs:
            n_sent += len(doc["sentences"])
        line = domain + " " + split + ": docs=" + str(len(docs)) + " sentences=" + str(n_sent)
        for window in WINDOWS_TO_BUILD:
            lens = []
            for ids in entry["windows"][window]["input_ids"]:
                lens.append(len(ids))
            if len(lens) > 0:
                lens.sort()
                line = line + " | w" + str(window) + " median/max len=" + str(lens[len(lens) // 2]) + "/" + str(lens[-1])
        line = line + " | chunks=" + str(len(chunk_list))
        print(line)

## 11. Verify

In [ ]:
problems = 0

for domain in DOMAINS:
    for split in SPLITS:
        entry = tokenized[domain][split]
        docs = entry["docs"]

        n_sent = 0
        for doc in docs:
            n_sent += len(doc["sentences"])

        # --- windows: row count, target tokens, labels ---
        for window in WINDOWS_TO_BUILD:
            w = entry["windows"][window]
            if len(w["input_ids"]) != n_sent:
                print("ROW COUNT MISMATCH", domain, split, window)
                problems += 1
            for r in range(len(w["input_ids"])):
                d = int(w["doc_idx"][r])
                i = int(w["sent_idx"][r])
                ids = w["input_ids"][r]
                start = int(w["target_start"][r])
                expected = docs[d]["sent_ids"][i]
                if not np.array_equal(ids[start:], expected):
                    problems += 1
                    if problems < 10:
                        print("TARGET MISMATCH", domain, split, window, d, i)
                if ids[0] != SEP_ID:
                    problems += 1
                if docs[d]["labels"] is not None:
                    if int(w["labels"][r]) != int(docs[d]["labels"][i]):
                        problems += 1
                        if problems < 10:
                            print("LABEL MISMATCH", domain, split, window, d, i)
            # window 0 must contain no neighbouring text
            if window == 0 and len(w["input_ids"]) > 0:
                first_len = len(w["input_ids"][0])
                expected_len = 1 + len(HEADER_IDS[domain]) + 1 + len(SENT_MARKER_IDS) + len(docs[0]["sent_ids"][0])
                if first_len != expected_len:
                    print("WINDOW-0 LENGTH UNEXPECTED", domain, split, first_len, expected_len)
                    problems += 1

        # --- chunks: each sentence appears exactly once as a target, in order ---
        seen = {}
        for chunk in entry["chunks"]:
            d = chunk["doc_idx"]
            if d not in seen:
                seen[d] = []
            ids = chunk["input_ids"]
            if len(ids) > DOC_CHUNK_LEN:
                problems += 1
                print("CHUNK TOO LONG", domain, split, len(ids))
            for k in range(len(chunk["end_pos"])):
                pos = int(chunk["end_pos"][k])
                if int(ids[pos]) != SEP_ID:
                    problems += 1
                s_idx = int(chunk["sent_idx"][k])
                # tokens just before the SEP must be this sentence
                sent = docs[d]["sent_ids"][s_idx]
                got = ids[pos - len(sent):pos]
                if not np.array_equal(got, sent):
                    problems += 1
                    if problems < 10:
                        print("CHUNK SENTENCE MISMATCH", domain, split, d, s_idx)
                if bool(chunk["is_target"][k]):
                    seen[d].append(s_idx)
                    if docs[d]["labels"] is not None:
                        if int(chunk["labels"][k]) != int(docs[d]["labels"][s_idx]):
                            problems += 1
                else:
                    if int(chunk["labels"][k]) != -100:
                        problems += 1
        for d in range(len(docs)):
            n = len(docs[d]["sentences"])
            got_list = seen.get(d, [])
            if got_list != list(range(n)):
                problems += 1
                print("CHUNK COVERAGE PROBLEM", domain, split, "doc", d)

        # --- test: output size contract ---
        if split == "test":
            for d in range(len(docs)):
                if len(docs[d]["sentences"]) != len(raw_docs[domain]["test"][d]["text"]):
                    problems += 1

print("Verification problems:", problems)
assert problems == 0, "Fix the problems above before using this data."
print("All checks passed.")

## 12. Save

In [ ]:
meta = {
    "hf_dataset_repo": HF_DATASET_REPO,
    "domains": DOMAINS,
    "label2id": label2id,
    "num_labels": num_labels,
    "tokenizer_dir": tok_dir,
    "tokenizer_vocab_size": len(tokenizer),
    "sep_id": SEP_ID,
    "pad_id": PAD_ID,
    "header_ids": HEADER_IDS,
    "before_ids": BEFORE_IDS,
    "after_ids": AFTER_IDS,
    "sent_marker_ids": SENT_MARKER_IDS,
    "max_sent_tok": MAX_SENT_TOK,
    "windows_built": WINDOWS_TO_BUILD,
    "budget_prev": BUDGET_PREV,
    "budget_next": BUDGET_NEXT,
    "doc_chunk_len": DOC_CHUNK_LEN,
    "doc_chunk_ctx": DOC_CHUNK_CTX,
    "normalise_punct": NORMALISE_PUNCT,
    "domain_id": DOMAIN_ID,
    "token_stats": tok_stats,
}

with open(os.path.join(OUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

audit_report["tokenisation"] = tok_stats
audit_report["roundtrip_bad"] = roundtrip_bad
audit_report["roundtrip_checked"] = roundtrip_checked
with open(os.path.join(OUT_DIR, "audit_report.json"), "w") as f:
    json.dump(audit_report, f, indent=2)

pkl_path = os.path.join(OUT_DIR, "tokenized_rr.pkl")
f = open(pkl_path, "wb")
pickle.dump({"meta": meta, "data": tokenized}, f, protocol=4)
f.close()
print("Saved", pkl_path, "%.1f MB" % (os.path.getsize(pkl_path) / 1e6))

# reload and spot check
f = open(pkl_path, "rb")
loaded = pickle.load(f)
f.close()
for domain in DOMAINS:
    for split in SPLITS:
        a = len(loaded["data"][domain][split]["docs"])
        b = len(tokenized[domain][split]["docs"])
        assert a == b
print("Reload OK.")

print()
for name in sorted(os.listdir(OUT_DIR)):
    print(" ", name)

## How the next notebooks should use this

```python
import pickle, sys
sys.path.insert(0, "/kaggle/input/<this-notebook-output>")   # for rr_utils.py
import rr_utils

blob = pickle.load(open("/kaggle/input/<this-notebook-output>/tokenized_rr.pkl", "rb"))
meta = blob["meta"]
train = blob["data"]["CL"]["train"]

w2 = train["windows"][2]           # window format, +-2 sentences
ids = w2["input_ids"][0]           # token ids of row 0
start = w2["target_start"][0]      # ids[start:] is the target sentence
label = w2["labels"][0]

chunk = train["chunks"][0]         # document-chunk format
# hidden states at chunk["end_pos"] classify the sentences chunk["sent_idx"];
# only rows where chunk["is_target"] is True count (labels -100 are context)
```
Other window sizes: `rr_utils.build_window_example(...)` on `docs[d]["sent_ids"]`.